# ⚡ CALOAD — Bộ công cụ đa phương tiện cho Colab

Tải file siêu tốc, nén video, tải YouTube, xử lý hàng loạt.

**Kho mã nguồn:** https://github.com/tpc-pascal/caload  
**Tác giả:** tpc-pascal

---
### Hướng dẫn nhanh
1. Chạy các cell theo thứ tự từ trên xuống dưới.
2. Dùng các module bên dưới để thực hiện tác vụ.
3. Đầu ra được lưu tại `output/`, tải xuống tại `downloads/`.

In [ ]:
# === CELL 1: Cài đặt dependencies ===
print("Đang cài đặt dependencies...")
!apt update -qq && apt install -y -qq aria2 ffmpeg p7zip-full unrar 2>&1 | tail -1
!pip install -q yt-dlp tqdm requests psutil 2>&1 | tail -1
print("Hoàn tất!")

In [ ]:
# === CELL 2: Khởi tạo ===
import os, sys, subprocess, json, re, time, shutil, glob, logging
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass, field
from typing import List, Optional, Callable

BASE = Path("/content/caload")
BASE.mkdir(parents=True, exist_ok=True)
DOWNLOADS = BASE / "downloads"
OUTPUT = BASE / "output"
TEMP = BASE / "temp"
LOGS = BASE / "logs"
for d in [DOWNLOADS, OUTPUT, TEMP, LOGS]:
    d.mkdir(exist_ok=True)

print("Sẵn sàng!")

In [ ]:
# === Config & GPUScanner ===

class Config:
    BASE = BASE
    DOWNLOADS = DOWNLOADS
    OUTPUT = OUTPUT
    TEMP = TEMP
    LOGS = LOGS

    @staticmethod
    def ensure_dirs():
        for d in [Config.DOWNLOADS, Config.OUTPUT, Config.TEMP, Config.LOGS]:
            d.mkdir(exist_ok=True)

    @staticmethod
    def aria2_opts() -> list:
        return [
            "--max-connection-per-server=16",
            "--split=32",
            "--min-split-size=1M",
            "--continue=true",
            "--max-concurrent-downloads=5",
            "--dir=" + str(Config.DOWNLOADS),
        ]

class GPUScanner:
    def __init__(self):
        self.nvidia_smi = shutil.which("nvidia-smi")
        self.gpus = self._scan()

    def _scan(self) -> list:
        if not self.nvidia_smi:
            return []
        try:
            r = subprocess.run(
                [self.nvidia_smi, "--query-gpu=name,memory.total,compute_cap",
                 "--format=csv,noheader,nounits"],
                capture_output=True, text=True, timeout=10
            )
            return [g.strip() for g in r.stdout.strip().split("\n") if g.strip()]
        except Exception:
            return []

    def available(self) -> bool:
        return len(self.gpus) > 0

    def summary(self) -> str:
        if not self.gpus:
            return "Không phát hiện GPU"
        lines = [f"Phát hiện {len(self.gpus)} GPU:"]
        for g in self.gpus:
            parts = g.split(", ")
            if len(parts) >= 2:
                lines.append(f"  - {parts[0]} ({parts[1]} MB)")
        return "\n".join(lines)

    def nvenc_available(self) -> bool:
        codecs = subprocess.run(
            ["ffmpeg", "-encoders"], capture_output=True, text=True
        ).stdout
        return "h264_nvenc" in codecs

gpu = GPUScanner()
print(gpu.summary())

In [ ]:
# === Aria2Downloader ===

class Aria2Downloader:
    def __init__(self, timeout: int = 0):
        self.timeout = timeout
        self.aria2 = shutil.which("aria2c")
        if not self.aria2:
            raise RuntimeError("aria2c not found, install it first")

    def download(self, url: str, output: Optional[str] = None, **kwargs) -> str:
        cmd = [self.aria2] + Config.aria2_opts()
        if output:
            cmd += ["--out=" + output]
        if self.timeout:
            cmd += ["--timeout=" + str(self.timeout)]
        cmd += [url]
        subprocess.run(cmd, check=True)
        return str(Config.DOWNLOADS)

    def batch(self, urls: List[str]) -> List[str]:
        results = []
        for url in urls:
            self.download(url)
            results.append(str(Config.DOWNLOADS))
        return results

dl = Aria2Downloader()

In [ ]:
# === YoutubeDownloader ===

class YoutubeDownloader:
    def __init__(self):
        self.ytdlp = shutil.which("yt-dlp") or "yt-dlp"

    def _run(self, *args) -> str:
        cmd = [self.ytdlp] + list(args) + ["-o", str(Config.DOWNLOADS / "%(title)s.%(ext)s")]
        subprocess.run(cmd, check=True)
        return str(Config.DOWNLOADS)

    def best(self, url: str) -> str:
        return self._run("-f", "bestvideo+bestaudio/best", "--merge-output-format", "mp4", url)

    def audio(self, url: str, fmt: str = "mp3") -> str:
        return self._run("-f", "bestaudio", "--extract-audio", "--audio-format", fmt, url)

    def subtitles(self, url: str, lang: str = "vi") -> str:
        return self._run("--write-subs", "--sub-langs", lang, "--skip-download", url)

    def info(self, url: str) -> dict:
        r = subprocess.run(
            [self.ytdlp, "--dump-json", url],
            capture_output=True, text=True, timeout=30
        )
        return json.loads(r.stdout)

    def thumbnail(self, url: str) -> str:
        return self._run("--write-thumbnail", "--skip-download", "--convert-thumbnails", "jpg", url)

yt = YoutubeDownloader()

In [ ]:
# === FFmpegProcessor ===

class FFmpegProcessor:
    PRESETS = {
        "fast": {"crf": 28, "preset": "fast", "desc": "Nén nhanh, chất lượng trung bình"},
        "balanced": {"crf": 23, "preset": "medium", "desc": "Cân bằng tốc độ/chất lượng"},
        "best": {"crf": 18, "preset": "slow", "desc": "Chất lượng cao nhất, chậm"},
    }

    def __init__(self, gpu_scanner: Optional[GPUScanner] = None):
        self.gpu = gpu_scanner or GPUScanner()
        self.ffmpeg = shutil.which("ffmpeg") or "ffmpeg"

    def _encoder(self) -> str:
        if self.gpu and self.gpu.nvenc_available():
            return "h264_nvenc"
        return "libx264"

    def compress(self, input_path: str, mode: str = "balanced", output_name: Optional[str] = None) -> str:
        preset = self.PRESETS.get(mode, self.PRESETS["balanced"])
        inp = Path(input_path)
        out = Config.OUTPUT / (output_name or f"{inp.stem}_compressed{inp.suffix}")
        encoder = self._encoder()
        cmd = [self.ffmpeg, "-i", str(inp)]
        if encoder == "h264_nvenc":
            cmd += ["-c:v", "h264_nvenc", "-preset", "p7", "-cq", str(preset["crf"])]
        else:
            cmd += ["-c:v", "libx264", "-preset", preset["preset"], "-crf", str(preset["crf"])]
        cmd += ["-c:a", "aac", "-movflags", "+faststart", "-y", str(out)]
        subprocess.run(cmd, check=True)
        return str(out)

    def convert(self, input_path: str, fmt: str = "mp4") -> str:
        inp = Path(input_path)
        out = Config.OUTPUT / f"{inp.stem}.{fmt}"
        subprocess.run([self.ffmpeg, "-i", str(inp), "-y", str(out)], check=True)
        return str(out)

ff = FFmpegProcessor(gpu)

In [ ]:
# === BatchProcessor, ResourceMonitor, ArchiveTool ===

class BatchProcessor:
    def __init__(self, max_workers: int = 4):
        self.max_workers = max_workers

    def run(self, items: list, func: Callable, desc: str = "Đang xử lý", **kwargs) -> list:
        results = []
        with ThreadPoolExecutor(max_workers=self.max_workers) as ex:
            futures = {ex.submit(func, item, **kwargs): item for item in items}
            for f in as_completed(futures):
                try:
                    results.append(f.result())
                except Exception as e:
                    print(f"Lỗi [{futures[f]}]: {e}")
                    results.append(None)
        return results

class ResourceMonitor:
    def summary(self) -> str:
        try:
            import psutil
            cpu = psutil.cpu_percent(interval=0.5)
            mem = psutil.virtual_memory()
            disk = psutil.disk_usage("/content")
            return (
                f"CPU: {cpu}% | RAM: {mem.percent}% "
                f"({mem.used//1024**3}/{mem.total//1024**3} GB) | "
                f"Disk: {disk.percent}% ({disk.used//1024**3}/{disk.total//1024**3} GB)"
            )
        except ImportError:
            return "psutil not available"

class ArchiveTool:
    def extract(self, path: str, dest: Optional[str] = None) -> str:
        p = Path(path)
        dest = dest or str(Config.OUTPUT / p.stem)
        ext = p.suffix.lower()
        if ext == ".zip":
            import zipfile
            with zipfile.ZipFile(path, "r") as z:
                z.extractall(dest)
        elif ext in (".7z", ".rar"):
            tool = "7z" if ext == ".7z" else "unrar"
            subprocess.run([tool, "x", path, f"-o{dest}", "-y"], check=True)
        return dest

    def compress(self, path: str, fmt: str = "zip", output_name: Optional[str] = None) -> str:
        p = Path(path)
        out = Config.OUTPUT / (output_name or f"{p.stem}.{fmt}")
        if fmt == "zip":
            import zipfile
            with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
                if p.is_dir():
                    for f in p.rglob("*"):
                        z.write(f, f.relative_to(p.parent))
                else:
                    z.write(p, p.name)
        elif fmt == "7z":
            subprocess.run(["7z", "a", str(out), str(p)], check=True)
        return str(out)

bp = BatchProcessor()
arc = ArchiveTool()
m = ResourceMonitor()

In [ ]:
# === DriveManager ===

class DriveManager:
    MOUNT = Path("/content/drive")

    def mount(self) -> bool:
        if self.MOUNT.exists():
            return True
        subprocess.run(["mkdir", "-p", "/content/drive"], check=True)
        from google.colab import drive
        drive.mount("/content/drive")
        return True

    def ensure(self) -> Path:
        d = self.MOUNT / "MyDrive/caload"
        d.mkdir(parents=True, exist_ok=True)
        return d

    def scan_videos(self, ext: tuple = (".mp4", ".avi", ".mkv", ".mov")) -> list:
        videos = []
        root = self.MOUNT / "MyDrive"
        if not root.exists():
            return videos
        for f in root.rglob("*"):
            if f.suffix.lower() in ext:
                videos.append(str(f))
        return sorted(videos)

drive = DriveManager()

In [ ]:
# === VideoTools, AudioTool, ImageTool, LoggerSystem ===

class VideoTools:
    def __init__(self, gpu_scanner: Optional[GPUScanner] = None):
        self.ffmpeg = shutil.which("ffmpeg") or "ffmpeg"
        self.gpu = gpu_scanner

    def merge(self, video: str, audio: str, output: Optional[str] = None) -> str:
        out = Config.OUTPUT / (output or f"merged_{Path(video).name}")
        subprocess.run(
            [self.ffmpeg, "-i", video, "-i", audio,
             "-c:v", "copy", "-c:a", "aac", "-y", str(out)],
            check=True
        )
        return str(out)

    def trim(self, input_path: str, start: str, duration: str) -> str:
        inp = Path(input_path)
        out = Config.OUTPUT / f"{inp.stem}_trim{inp.suffix}"
        subprocess.run(
            [self.ffmpeg, "-i", str(inp), "-ss", start,
             "-t", duration, "-c", "copy", "-y", str(out)],
            check=True
        )
        return str(out)

    def metadata(self, input_path: str) -> dict:
        r = subprocess.run(
            [self.ffmpeg, "-i", input_path],
            capture_output=True, text=True, stderr=subprocess.PIPE
        )
        info = {}
        for line in r.stderr.split("\n"):
            if "Duration:" in line:
                m = re.search(r"Duration: (\d+:\d+:\d+\.\d+)", line)
                if m:
                    info["duration"] = m.group(1)
            elif "Stream #" in line:
                if "Video:" in line:
                    m = re.search(r"(\d+x\d+)", line)
                    if m:
                        info["resolution"] = m.group(1)
                    m2 = re.search(r"Video: (\w+)", line)
                    if m2:
                        info["vcodec"] = m2.group(1)
                elif "Audio:" in line:
                    m = re.search(r"Audio: (\w+)", line)
                    if m:
                        info["acodec"] = m.group(1)
        return info

class AudioTool:
    def __init__(self):
        self.ffmpeg = shutil.which("ffmpeg") or "ffmpeg"

    def extract(self, video_path: str, fmt: str = "mp3", sample_rate: str = "44100") -> str:
        inp = Path(video_path)
        out = Config.OUTPUT / f"{inp.stem}.{fmt}"
        subprocess.run(
            [self.ffmpeg, "-i", str(inp), "-vn",
             "-c:a", "libmp3lame" if fmt == "mp3" else "aac",
             "-ar", sample_rate, "-y", str(out)],
            check=True
        )
        return str(out)

class ImageTool:
    def optimize(self, input_path: str, quality: int = 85) -> str:
        inp = Path(input_path)
        out = Config.OUTPUT / f"{inp.stem}_optimized{inp.suffix}"
        subprocess.run(
            ["ffmpeg", "-i", str(inp),
             "-q:v", str(int(quality / 10)), "-y", str(out)],
            check=True
        )
        return str(out)

class LoggerSystem:
    def __init__(self, name: str = "caload"):
        self.logger = logging.getLogger(name)
        self.logger.setLevel(logging.INFO)
        fh = logging.FileHandler(Config.LOGS / f"{name}.log")
        fh.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
        self.logger.addHandler(fh)

    def info(self, msg: str):
        self.logger.info(msg)

    def error(self, msg: str):
        self.logger.error(msg)

vt = VideoTools(gpu)
at = AudioTool()
it = ImageTool()
log = LoggerSystem()

print("Sẵn sàng!")

---
### Ví dụ sử dụng

Chạy từng cell bên dưới để thực hiện tác vụ.

In [ ]:
# Tải file (aria2c)
path = dl.download("https://example.com/file.zip")
print(f"Đã lưu: {path}")

In [ ]:
# YouTube chất lượng cao nhất
path = yt.best("https://youtube.com/watch?v=...")
print(f"Đã lưu: {path}")

In [ ]:
# YouTube chỉ lấy audio
path = yt.audio("https://youtube.com/watch?v=...", fmt="mp3")
print(f"Đã lưu: {path}")

In [ ]:
# Nén video (tự động dùng GPU NVENC nếu có)
path = ff.compress("/content/video.mp4", mode="balanced")
print(f"Đã lưu: {path}")

In [ ]:
# Nén hàng loạt video
videos = glob.glob("/content/drive/MyDrive/videos/*.mp4")
results = bp.run(videos, ff.compress, desc="Đang nén")
print(f"Hoàn tất: {len([r for r in results if r])}/{len(videos)}")

In [ ]:
# Google Drive
drive.mount()
drive.ensure()
videos = drive.scan_videos()
print(f"Tìm thấy {len(videos)} video trong Drive")
for v in videos[:5]:
    print(f"  {v}")

In [ ]:
# Giám sát tài nguyên
print(m.summary())

In [ ]:
# Giải nén archive
arc.extract("/content/archive.zip")

In [ ]:
# Cắt video
path = vt.trim("/content/video.mp4", start="00:01:00", duration="00:00:30")
print(f"Đã lưu: {path}")

In [ ]:
# Tách audio từ video
path = at.extract("/content/video.mp4", fmt="mp3")
print(f"Đã lưu: {path}")

In [ ]:
# YouTube tải thumbnail
path = yt.thumbnail("https://youtube.com/watch?v=...")
print(f"Đã lưu: {path}")

In [ ]:
# Nén file/folder thành ZIP
path = arc.compress("/content/folder", fmt="zip")
print(f"Đã lưu: {path}")